# `hopfield_tracking` — Denby (1988)'s Hopfield-network track finder

A from-scratch, didactic reimplementation of the continuous Hopfield
network from B. Denby, *"Neural networks and cellular automata in
experimental high energy physics"* (1988), section 8 — one of the first
applications of a neural network to particle-track reconstruction. It
works on plain `(x, y)` hit points; it has no notion of `detector2d` layers
at all.

- **Neurons** are candidate track *segments*: `(i, j)` from point `i` to
  point `j`. A valid track is a non-bifurcating chain of "on" segments.
- **Dynamics**: `tau dv/dt = sum_j T_ij f(v_j) - v_i`, a piecewise-linear
  sigmoid `f`. **Energy** `E = -1/2 sum_ij T_ij f(v_i) f(v_j)` decreases as
  the network relaxes.
- **Coefficients `T_ij`**: type 1 (segments sharing an endpoint) reward a
  smooth, short continuation; type 2 (segments that don't share a point but
  lie nearby) reward a good 4-point circle fit; inhibition penalizes
  segments competing for the same endpoint.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

from hopfield_tracking.network import build_segments, mean_consecutive_hit_distance
from hopfield_tracking.coefficients import build_weight_matrix, DEFAULT_TYPE1_SCALE
from hopfield_tracking.dynamics import relax, DEFAULT_GAIN, DEFAULT_INIT_SPREAD
from hopfield_tracking.extract import on_segments, chain_tracks, score_against_truth
from hopfield_tracking.vis import plot_iterations


## A clean test case: three widely-separated straight tracks

Three tracks fan out from a common vertex at well-separated angles (~23
degrees apart) — the same clean case used to calibrate this package's
defaults in the first place.


In [ ]:
rows = []
for particle_id, phi in enumerate([1.2, 1.6, 2.0]):  # radians
    for k in range(1, 9):
        rows.append(dict(particle_id=particle_id, x=k * 60.0 * np.cos(phi), y=k * 60.0 * np.sin(phi)))
hits = pd.DataFrame(rows)
print(f"{len(hits)} hits, {hits['particle_id'].nunique()} true tracks")


## Building candidate segments and the weight matrix

`R_c` (the candidate-neighbor cutoff) and `r_scale` are calibrated from the
data's own mean consecutive-hit distance `<r>`, following the paper.


In [ ]:
mean_r = mean_consecutive_hit_distance(hits)
r_c = 1.5 * mean_r
segments = build_segments(hits["x"].to_numpy(), hits["y"].to_numpy(), r_c)
print(f"<r> = {mean_r:.1f}, R_c = {r_c:.1f}, {len(segments)} candidate segments")

t = build_weight_matrix(segments, r_c=r_c, r_scale=mean_r, inhibition=-0.5, use_type2=False)


## Relaxing the network

Starting from a narrow random band around the sigmoid's unstable center
(not spread across `[0, 1]`), the dynamics amplify the true chain's
segments and suppress the rest — converging in well under 100 iterations.


In [ ]:
history = relax(t, n_iterations=100, gain=DEFAULT_GAIN, rng=np.random.default_rng(0), energy_tol=1e-9)
print(f"converged after {len(history)} iterations")

active = on_segments(segments, history.f_v[-1])
chains = chain_tracks(active)
score = score_against_truth(chains, dict(enumerate(hits["particle_id"])))
print(score)


In [ ]:
fig = plot_iterations(hits[["x", "y"]].to_numpy(), segments, history)


All 3 tracks reconstruct exactly, segment-for-segment — proof the mechanism
(coefficients + dynamics + extraction) is correct on a clean, well-separated
case. Running the same algorithm on the paper's own crowded 4-track event
(where all tracks share one vertex) instead reproduces its documented
imperfection: 2 of 4 tracks exact, the other 2 correct along their outer
hits with confusion right at the shared vertex — see the next notebook,
`10_denby_1988_recreation`.


## Two implicit findings that turned out to matter

The paper's own coefficients aren't quite enough on their own — getting
this to actually converge to the right answer (not the trivial "everything
off" solution) took two structural fixes, demonstrated directly below
rather than just asserted.

**1. `type1_scale` must exceed ~1.** The paper's own type-1 formula gives an
interior segment on a straight chain a coefficient of exactly 0.5 to each
neighbor — only *neutrally* stable, so a finite chain's end segments decay
and the weakness cascades inward until the whole chain vanishes. Watch what
happens if we relax with `type1_scale=1.0` instead of the calibrated
default (`{DEFAULT_TYPE1_SCALE}`):


In [ ]:
t_understrength = build_weight_matrix(
    segments, r_c=r_c, r_scale=mean_r, inhibition=-0.5, use_type2=False, type1_scale=1.0,
)
history_understrength = relax(t_understrength, n_iterations=100, gain=DEFAULT_GAIN,
                               rng=np.random.default_rng(0), energy_tol=1e-9)
active_understrength = on_segments(segments, history_understrength.f_v[-1])
print(f"type1_scale=1.0:  {len(active_understrength)} active segments (decayed to the trivial solution)")
print(f"type1_scale={DEFAULT_TYPE1_SCALE} (default): {len(active)} active segments (the correct 3 tracks)")


**2. The initial `v` should be narrow (`DEFAULT_INIT_SPREAD = 0.1`), not
spread across `[0, 1]`.** A wide random start combined with a steep sigmoid
gain just saturates every neuron to 0 or 1 based on its *initial random
sign*, before the network's own structure gets a chance to matter — this is
handled internally by `relax`'s `init_spread` parameter and is why the
result above is reproducible rather than gain-dependent noise.
